# 06 — Applications: Impact Zones

This notebook brings together everything from the module:

- **Point buffers** → blast radii around targets
- **Line buffers** → danger corridor along a trajectory
- **Containment check** → which locations fall inside a given zone?
- **Combined view** → path + impact zones on a single map

The result is a complete zone-based analysis: given a set of targets and a trajectory, identify what is within range of what.

## Setup

In [ ]:
import json
import math
from pathlib import Path
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps

DATA_DIR = Path("data")

with open(DATA_DIR / "targets.geojson") as f:
    targets_fc = json.load(f)

paths_file = Path("../09-Intersections/data/paths.geojson")
if paths_file.exists():
    with open(paths_file) as f:
        paths_fc = json.load(f)
else:
    paths_fc = {
        "type": "FeatureCollection",
        "features": [
            {"type": "Feature", "properties": {"name": "Alpha"},
             "geometry": {"type": "LineString",
                          "coordinates": [[-77.009, 38.889], [51.388, 35.695]]}},
            {"type": "Feature", "properties": {"name": "Delta"},
             "geometry": {"type": "LineString",
                          "coordinates": [[37.617, 55.755], [46.675, 24.688]]}},
        ]
    }


# --- Core geometry helpers ---

def destination_point(lon, lat, bearing_deg, distance_km):
    R = 6371.0
    d = distance_km / R
    phi1 = math.radians(lat)
    lam1 = math.radians(lon)
    theta = math.radians(bearing_deg)
    phi2 = math.asin(math.sin(phi1)*math.cos(d) + math.cos(phi1)*math.sin(d)*math.cos(theta))
    lam2 = lam1 + math.atan2(math.sin(theta)*math.sin(d)*math.cos(phi1),
                              math.cos(d) - math.sin(phi1)*math.sin(phi2))
    return [math.degrees(lam2), math.degrees(phi2)]

def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def point_buffer(lon, lat, radius_km, n_points=64):
    ring = [destination_point(lon, lat, 360.0*i/n_points, radius_km) for i in range(n_points)]
    ring.append(ring[0])
    return {"type": "Polygon", "coordinates": [ring]}

def buffer_feature(lon, lat, radius_km, name="", n_points=64):
    return {
        "type": "Feature",
        "properties": {"name": name, "radius_km": radius_km},
        "geometry": point_buffer(lon, lat, radius_km, n_points)
    }

def sample_linestring(coords, n_samples):
    n_segs = len(coords) - 1
    per_seg = max(1, n_samples // n_segs)
    points = []
    for i in range(n_segs):
        p1, p2 = coords[i], coords[i + 1]
        for j in range(per_seg):
            t = j / per_seg
            points.append([p1[0] + t*(p2[0]-p1[0]), p1[1] + t*(p2[1]-p1[1])])
    points.append(coords[-1])
    return points

def line_buffer(path_feature, radius_km, n_samples=40, n_points=32):
    coords  = path_feature["geometry"]["coordinates"]
    samples = sample_linestring(coords, n_samples)
    name    = path_feature["properties"].get("name", "")
    return {
        "type": "FeatureCollection",
        "features": [
            {"type": "Feature",
             "properties": {"path": name, "radius_km": radius_km},
             "geometry": point_buffer(pt[0], pt[1], radius_km, n_points)}
            for pt in samples
        ]
    }

def target_coords(name):
    feat = next(f for f in targets_fc["features"] if f["properties"]["name"] == name)
    return feat["geometry"]["coordinates"]

print(f"Loaded {len(targets_fc['features'])} targets, {len(paths_fc['features'])} paths.")

## Blast Radius Zones Around a Target

Three concentric rings model different intensity levels of an impact event. Each ring represents a zone with distinct effects — the boundaries are policy or physics-based thresholds, not arbitrary choices.

```
Lethal zone   (50 km)  — immediate, severe
Damaging zone (150 km) — structural damage, casualties possible
Warning zone  (350 km) — detectable effects, evacuation advisable
```

In [ ]:
tehran = target_coords("Tehran")

blast_zones = [
    {"radius_km": 50,  "label": "Lethal",   "color": "#e74c3c", "opacity": 0.55},
    {"radius_km": 150, "label": "Damaging", "color": "#e67e22", "opacity": 0.30},
    {"radius_km": 350, "label": "Warning",  "color": "#f1c40f", "opacity": 0.15},
]

m_blast = Map(center=(tehran[1], tehran[0]), zoom=4, basemap=basemaps.CartoDB.Positron)

for zone in reversed(blast_zones):   # outer first → inner on top
    feat = buffer_feature(*tehran, radius_km=zone["radius_km"], name=zone["label"])
    m_blast.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        name=f"{zone['label']} ({zone['radius_km']} km)",
        style={"color": zone["color"], "fillColor": zone["color"],
               "fillOpacity": zone["opacity"], "weight": 2, "opacity": 0.9}
    ))

# Target point
m_blast.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [
        {"type": "Feature", "properties": {"name": "Tehran"},
         "geometry": {"type": "Point", "coordinates": tehran}}
    ]},
    style={"color": "#2c3e50", "fillColor": "#2c3e50", "fillOpacity": 1.0, "weight": 1}
))
m_blast.add(LayersControl())

print("Blast zones around Tehran: 50 / 150 / 350 km")
m_blast

## Trajectory Corridor

A line buffer around a missile path defines the **danger corridor** — the airspace and surface within a given distance of the trajectory at any point along its flight.

In [ ]:
alpha = next(f for f in paths_fc["features"] if f["properties"]["name"] == "Alpha")

corridor = line_buffer(alpha, radius_km=200, n_samples=60)

m_corridor = Map(center=(40, 5), zoom=3, basemap=basemaps.CartoDB.Positron)

m_corridor.add(GeoJSON(
    data=corridor,
    name="Alpha corridor (200 km)",
    style={"color": "#2980b9", "fillColor": "#2980b9", "fillOpacity": 0.12, "weight": 0}
))
m_corridor.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [alpha]},
    name="Alpha path",
    style={"color": "#2980b9", "weight": 2.5, "opacity": 0.9}
))

print(f"Alpha corridor: {len(corridor['features'])} circle polygons, 200 km radius")
m_corridor

## Containment Check: Which Targets Fall Within a Blast Zone?

A buffer tells us the *zone* — but we often need to ask: is a specific point inside it? For a circular buffer, the answer is simple:

```python
inside = haversine_km(point, center) <= radius_km
```

We can run this check for every target against every other target's blast radius to find **mutual exposure** — targets close enough to affect one another.

In [ ]:
# Extract all targets as (name, lon, lat)
targets = [
    (f["properties"]["name"],
     f["geometry"]["coordinates"][0],
     f["geometry"]["coordinates"][1])
    for f in targets_fc["features"]
]

BLAST_RADIUS_KM = 1500   # hypothetical long-range blast zone

print(f"Checking which targets fall within {BLAST_RADIUS_KM} km of each other:\n")
print(f"{'Source':10}  {'Target':10}  {'Distance (km)':>14}  {'Within zone?':>13}")
print("─" * 55)

for src_name, src_lon, src_lat in targets:
    for tgt_name, tgt_lon, tgt_lat in targets:
        if src_name == tgt_name:
            continue
        dist = haversine_km(src_lon, src_lat, tgt_lon, tgt_lat)
        inside = dist <= BLAST_RADIUS_KM
        marker = "YES" if inside else "no"
        print(f"  {src_name:10}  {tgt_name:10}  {dist:>13.0f}  {marker:>13}")

## Combined View: Path + Impact Zones

The most complete analysis overlays:
1. The trajectory (line)
2. The trajectory corridor (line buffer)
3. The blast zones at the destination target (point buffer)

This lets you see the full picture: where the missile travels, what airspace is threatened, and what happens at the endpoint.

In [ ]:
# Alpha path: Washington D.C. → Tehran
# Corridor: 150 km around the path
# Blast zones at Tehran: 50 / 150 / 300 km

flight_corridor = line_buffer(alpha, radius_km=150, n_samples=60)

tehran_zones = [
    {"radius_km": 50,  "label": "Lethal",   "color": "#e74c3c", "opacity": 0.60},
    {"radius_km": 150, "label": "Damaging", "color": "#e67e22", "opacity": 0.35},
    {"radius_km": 300, "label": "Warning",  "color": "#f1c40f", "opacity": 0.18},
]

m_combined = Map(center=(40, 10), zoom=3, basemap=basemaps.CartoDB.Positron)

# 1. Flight corridor (bottom — covers large area, keep very transparent)
m_combined.add(GeoJSON(
    data=flight_corridor,
    name="Flight corridor (150 km)",
    style={"color": "#2980b9", "fillColor": "#2980b9", "fillOpacity": 0.08, "weight": 0}
))

# 2. Blast zones at destination (outer → inner)
for zone in reversed(tehran_zones):
    feat = buffer_feature(*tehran, radius_km=zone["radius_km"], name=zone["label"])
    m_combined.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        name=f"Tehran {zone['label']} ({zone['radius_km']} km)",
        style={"color": zone["color"], "fillColor": zone["color"],
               "fillOpacity": zone["opacity"], "weight": 1.5, "opacity": 0.9}
    ))

# 3. Path line (on top)
m_combined.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [alpha]},
    name="Alpha path",
    style={"color": "#2c3e50", "weight": 2, "opacity": 0.8}
))

m_combined.add(LayersControl())
print("Combined map: Alpha trajectory + corridor + Tehran blast zones")
m_combined

## Exercise A

Build a **blast radius map for Riyadh** with three zones: 75 km (lethal), 200 km (damaging), 400 km (warning).

1. Display all three zones with appropriate colors and opacities.
2. Using a containment check (`haversine_km`), determine which of the other three targets (Tehran, Honolulu, Madrid) fall within Riyadh's 400 km warning zone.
3. Print the distance from Riyadh to each of the other targets and label each as inside or outside the zone.

In [ ]:
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps

riyadh = target_coords("Riyadh")

# 1. Three blast zones around Riyadh (75, 200, 400 km)
# 2. Containment check: which targets are within 400 km of Riyadh?
# 3. Print distances and inside/outside labels
# Your code here

## Exercise B

Create a **complete trajectory analysis** for the **Delta path** (Moscow → Riyadh).

1. Draw a 100 km flight corridor along the Delta path.
2. Add blast zones (50 km, 150 km, 300 km) at the **destination** (Riyadh).
3. Add a 300 km blast zone at the **origin** (Moscow) to show the launch vicinity.
4. Display all layers on one map with a `LayersControl`.
5. Print the total path length (haversine from start to end) in km.

In [ ]:
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps

delta = next(f for f in paths_fc["features"] if f["properties"]["name"] == "Delta")
moscow = delta["geometry"]["coordinates"][0]
riyadh = target_coords("Riyadh")

# 1. 100 km corridor along Delta
# 2. 50/150/300 km blast zones at Riyadh (destination)
# 3. 300 km blast zone at Moscow (origin)
# 4. Display all with LayersControl
# 5. Print path length in km
# Your code here

---

## Check Your Understanding

**1.** How do buffers help model real-world impact zones, and what is the limit of that model?

**2.** Why is it important to choose the buffer radius carefully rather than using an arbitrary value?

```python
# No code needed — answer in your own words
```

## Next

In [11 — Point in Polygon](../../11-Point_In_Polygon/00-Concepts.ipynb), we move from distance-based containment to geometric containment — testing whether a point falls inside an arbitrary polygon using the ray-casting algorithm.